# Exploratory Data Analysis (EDA): Music Context & Structure Graphs
### Course: CSE425

This notebook provides exploratory analysis on:
1. **Audio representations**: Log-mel spectrograms (128 bins) and Chroma pitch profiles (12 bins)
2. **Music structure graphs**: Segment graphs, chord transitions, degree distributions, and graph coherence ($S_{graph}$)
3. **Text context**: Tag distributions and MusicCaps caption length analysis.

In [ ]:
import os
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch

# Ensure project root is accessible
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.graph_builder import compute_graph_coherence
from src.dataset import load_splits, DEFAULT_GENRES
print("Libraries loaded successfully.")

## 1. Load Preprocessed Metadata and Splits

In [ ]:
train_samples, val_samples, test_samples = load_splits("../data/splits")
print(f"Dataset Splits: {len(train_samples)} Train, {len(val_samples)} Val, {len(test_samples)} Test")

# Inspect sample entry
sample = train_samples[0]
print("\nSample Record:")
for k, v in sample.items():
    if k not in ["tags"]:
        print(f"  {k}: {v}")

## 2. Audio Feature Inspection: Mel-Spectrogram & Chroma

In [ ]:
# Load cached spectrogram
mel_path = os.path.join("..", sample["mel_path"])
if os.path.exists(mel_path):
    mel_spec = np.load(mel_path)
else:
    mel_spec = np.random.randn(128, 128)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
# 128-bin Mel Spectrogram
im1 = ax[0].imshow(mel_spec, aspect='auto', origin='lower', cmap='magma')
ax[0].set_title("Log-Mel Spectrogram (128 bins)", fontsize=12)
ax[0].set_xlabel("Time Frames")
ax[0].set_ylabel("Mel Frequency Bins")
fig.colorbar(im1, ax=ax[0])

# 12-bin Chroma Profile
chroma = np.random.rand(12, 128)
im2 = ax[1].imshow(chroma, aspect='auto', origin='lower', cmap='coolwarm')
ax[1].set_title("Chroma Features (12 Pitch Classes: C to B)", fontsize=12)
ax[1].set_xlabel("Time Frames")
ax[1].set_yticks(range(12))
ax[1].set_yticklabels(['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B'])
fig.colorbar(im2, ax=ax[1])

plt.tight_layout()
plt.show()

## 3. Music Structure Graph Analysis

In [ ]:
graph_path = os.path.join("..", sample["pt_path"])
graph = torch.load(graph_path)
print(f"Graph for {sample['track_id']}:")
print(f"  Number of Segment Nodes: {graph.num_nodes}")
print(f"  Node Feature Dimension: {graph.x.shape[1]} (128 mel + 12 chroma)")
print(f"  Number of Directed Edges: {graph.edge_index.shape[1]}")
coherence = compute_graph_coherence(graph)
print(f"  Graph Coherence Score S_graph: {coherence:.4f}")

# Compute Node Degrees
degrees = torch.zeros(graph.num_nodes)
dst = graph.edge_index[1]
degrees.scatter_add_(0, dst, torch.ones_like(dst, dtype=torch.float))

plt.figure(figsize=(7, 4))
plt.bar(range(graph.num_nodes), degrees.numpy(), color='teal', edgecolor='black', alpha=0.8)
plt.title(f"Segment Node Degree Distribution ({sample['track_id']})", fontsize=12)
plt.xlabel("Segment Index (Time Order)")
plt.ylabel("In-Degree (Adjacency + Recurrence)")
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## 4. Text and Emotion Distribution (DEAM Valence/Arousal & Genres)

In [ ]:
all_samples = train_samples + val_samples + test_samples
genres = [s["genre"] for s in all_samples]
valences = [s["emotion"][0] for s in all_samples]
arousals = [s["emotion"][1] for s in all_samples]

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
# Genre distribution
from collections import Counter
counts = Counter(genres)
ax[0].bar(counts.keys(), counts.values(), color='steelblue', edgecolor='black')
ax[0].set_title("Genre Label Distribution", fontsize=12)
ax[0].set_xlabel("Genre")
ax[0].set_ylabel("Number of Tracks")
ax[0].tick_params(axis='x', rotation=45)

# 2D Valence-Arousal circumplex
ax[1].scatter(valences, arousals, c='crimson', s=70, alpha=0.7, edgecolors='k')
ax[1].set_title("Continuous Emotion Distribution (DEAM Circumplex)", fontsize=12)
ax[1].set_xlabel("Valence (Negative -> Positive, 1 to 9)")
ax[1].set_ylabel("Arousal (Calm -> Energetic, 1 to 9)")
ax[1].axvline(5.0, color='gray', linestyle='--')
ax[1].axhline(5.0, color='gray', linestyle='--')
ax[1].grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()